# Build Node View

This notebook adds stable primary keys and materializes the node view used by the lineage experience.

In [ ]:
import time
import pandas as pd
from pyspark.sql.types import StringType, StructField, StructType

_phase_timers = {}
_table_cache = {}

def start_phase(name):
    _phase_timers[name] = time.perf_counter()
    print(f"[timer] start {name}")

def end_phase(name):
    start = _phase_timers.get(name)
    if start is None:
        print(f"[timer] {name}: missing start")
        return
    elapsed = time.perf_counter() - start
    print(f"[timer] {name}: {elapsed:.2f}s")

def load_table(table_name):
    try:
        return spark.table(table_name).toPandas()
    except Exception:
        return pd.DataFrame()

def get_table(table_name):
    if table_name not in _table_cache:
        table_start = time.perf_counter()
        _table_cache[table_name] = load_table(table_name)
        print(f"[timer] load_table:{table_name}: {time.perf_counter() - table_start:.2f}s")
    return _table_cache[table_name]

def ensure_columns(df, required_columns):
    result = df.copy() if hasattr(df, 'copy') else pd.DataFrame(df)
    if result.empty and len(result.columns) == 0:
        return pd.DataFrame(columns=required_columns)
    for column_name in required_columns:
        if column_name not in result.columns:
            result[column_name] = None
    return result

def normalize_for_spark_write(df, required_columns=None):
    final_df = ensure_columns(df, required_columns or []) if required_columns else (df.copy() if hasattr(df, 'copy') else pd.DataFrame(df))
    if final_df.empty and len(final_df.columns) == 0:
        return pd.DataFrame(columns=required_columns or [])
    for column_name in final_df.columns:
        final_df[column_name] = final_df[column_name].map(lambda value: None if pd.isna(value) or value == 'nan' else str(value))
    if required_columns:
        final_df = final_df[required_columns]
    return final_df

def write_delta_table(df, table_name):
    final_df = normalize_for_spark_write(df)
    if final_df.empty and len(final_df.columns) == 0:
        return
    schema = StructType([StructField(column_name, StringType(), True) for column_name in final_df.columns])
    spark.createDataFrame(final_df, schema=schema).write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable(table_name)

StatementMeta(, 2072f8f5-3a72-42ed-8bf2-6bfed48b828f, 6, Finished, Available, Finished, False)

In [ ]:
table_names = [
    't_fabric_artifacts',
    't_dataset_columns',
    't_dataset_tables',
    't_dataset_measures',
    't_dataset_relations',
    't_dataset_dependencies',
    't_report_metadata',
    't_report_pages',
    't_report_visuals',
    't_report_semantic_objects',
    't_lakehouse_metadata',
    't_lakehouse_tables',
    't_lakehouse_columns',
    't_warehouse_metadata',
    't_warehouse_tables',
    't_warehouse_columns'
]

print(f"Registered tables for lazy loading: {', '.join(table_names)}")
print('Tables will load on first use inside the node builder loop.')

StatementMeta(, 2072f8f5-3a72-42ed-8bf2-6bfed48b828f, 7, Finished, Available, Finished, False)

Loaded tables: t_fabric_artifacts, t_dataset_columns, t_dataset_tables, t_dataset_measures, t_dataset_relations, t_dataset_dependencies, t_report_metadata, t_report_pages, t_report_visuals, t_report_semantic_objects, t_lakehouse_metadata, t_lakehouse_tables, t_lakehouse_columns, t_warehouse_metadata, t_warehouse_tables, t_warehouse_columns


In [8]:
def non_empty_text(value):
    if value is None:
        return None
    text = str(value).strip()
    if not text or text.lower() == 'nan':
        return None
    return text

def create_nodes(df, node_id_col, parent_col_or_func, name_col_or_func, node_type_value):
    """Base helper for creating node frames with common structure"""
    if df is None or df.empty:
        return None
    
    nodes = df.copy()
    nodes['node_id'] = nodes[node_id_col]
    nodes['parent_node'] = parent_col_or_func(nodes) if callable(parent_col_or_func) else parent_col_or_func
    nodes['node_name'] = name_col_or_func(nodes) if callable(name_col_or_func) else nodes[name_col_or_func]
    nodes['node_type'] = node_type_value(nodes) if callable(node_type_value) else node_type_value
    
    return nodes[['node_id', 'parent_node', 'node_name', 'node_type', 'workspace_id']]

# Individual node builder functions (easy to debug and extend)
def build_artifact_nodes(df):
    """Build nodes from Fabric artifacts"""
    return create_nodes(df, 'id', None, 'display_name', lambda d: d['type'].str.lower())

def build_dataset_table_nodes(df):
    """Build nodes from dataset tables"""
    return create_nodes(df, 'table_pk', lambda d: d['dataset_id'], 'name', 'table')

def build_column_nodes(df):
    """Build nodes from dataset columns"""
    return create_nodes(
        df, 
        'column_pk',
        lambda d: d['table_name'].astype(str) + '-' + d['dataset_id'].astype(str),
        lambda d: d['column_name'].map(non_empty_text).fillna(d['column_pk'].astype(str)),
        'column'
    )

def build_measure_nodes(df):
    """Build nodes from dataset measures"""
    return create_nodes(
        df,
        'measure_pk',
        lambda d: d['table_name'].astype(str) + '-' + d['dataset_id'].astype(str),
        lambda d: d['measure_name'].map(non_empty_text).fillna(d['measure_pk'].astype(str)),
        'measure'
    )

def build_report_nodes(df):
    """Build nodes from report metadata"""
    return create_nodes(
        df,
        'report_id',
        None,
        lambda d: d['report_name'].map(non_empty_text).fillna(d['report_id'].astype(str)),
        'report'
    )

def build_page_nodes(df):
    """Build nodes from report pages"""
    return create_nodes(
        df,
        'page_pk',
        lambda d: d['report_id'],
        lambda d: d['page_display_name'].map(non_empty_text).fillna(d['page_name'].map(non_empty_text)).fillna(d['page_pk'].astype(str)),
        'page'
    )

def build_visual_nodes(df):
    """Build nodes from report visuals"""
    def format_visual_name(d):
        return d.apply(
            lambda row: f"{non_empty_text(row.get('display_type') or row.get('type') or row.get('visual_type')) or 'Visual'}: "
                       f"{non_empty_text(row.get('title') or row.get('visual_title') or row.get('display_name') or row.get('visual_name') or row.get('name')) or row.get('visual_name') or row.get('visual_pk')}",
            axis=1
        )
    return create_nodes(
        df,
        'visual_pk',
        lambda d: d['page_name'].astype(str) + '-' + d['report_id'].astype(str),
        format_visual_name,
        'visual'
    )

def build_lakehouse_nodes(df):
    """Build nodes from lakehouse metadata"""
    return create_nodes(
        df,
        'id',
        None,
        lambda d: d['display_name'].map(non_empty_text).fillna(d['name'].map(non_empty_text)).fillna(d['id'].astype(str)),
        'lakehouse'
    )

def build_lakehouse_table_nodes(df):
    """Build nodes from lakehouse tables"""
    return create_nodes(
        df,
        'lakehouse_table_pk',
        lambda d: d['lakehouse_id'],
        lambda d: d['table_name'].map(non_empty_text).fillna(d['lakehouse_table_pk'].astype(str)),
        'lakehouse_table'
    )

def build_warehouse_nodes(df):
    """Build nodes from warehouse metadata"""
    return create_nodes(
        df,
        'id',
        None,
        lambda d: d['display_name'].map(non_empty_text).fillna(d['name'].map(non_empty_text)).fillna(d['id'].astype(str)),
        'warehouse'
    )

def build_warehouse_table_nodes(df):
    """Build nodes from warehouse tables"""
    return create_nodes(
        df,
        'warehouse_table_pk',
        lambda d: d['warehouse_id'],
        lambda d: d['table_name'].map(non_empty_text).fillna(d['warehouse_table_pk'].astype(str)),
        'warehouse_table'
    )

StatementMeta(, 2072f8f5-3a72-42ed-8bf2-6bfed48b828f, 12, Finished, Available, Finished, False)

In [ ]:
# Registry of node builders: pairs each table with its builder function
node_builders = [
    ('t_fabric_artifacts', build_artifact_nodes),
    ('t_dataset_tables', build_dataset_table_nodes),
    ('t_dataset_columns', build_column_nodes),
    ('t_dataset_measures', build_measure_nodes),
    ('t_report_metadata', build_report_nodes),
    ('t_report_pages', build_page_nodes),
    ('t_report_visuals', build_visual_nodes),
    ('t_lakehouse_metadata', build_lakehouse_nodes),
    ('t_lakehouse_tables', build_lakehouse_table_nodes),
    ('t_warehouse_metadata', build_warehouse_nodes),
    ('t_warehouse_tables', build_warehouse_table_nodes),
]

start_phase('build_nodes')

# Build all node frames
node_frames = []
for table_name, builder_func in node_builders:
    table_start = time.perf_counter()
    df = get_table(table_name)
    if df is None or df.empty:
        print(f"Skipping {table_name}: no rows")
        print(f"[timer] node_builder:{table_name}: {time.perf_counter() - table_start:.2f}s")
        continue

    try:
        node_frame = builder_func(df)
        if node_frame is not None and not node_frame.empty:
            node_frames.append(node_frame)
            print(f"Built nodes from {table_name}: {len(node_frame)} rows")
        else:
            print(f"Skipping {table_name}: builder returned empty")
    except Exception as e:
        print(f"Error building nodes from {table_name}: {e}")
    finally:
        print(f"[timer] node_builder:{table_name}: {time.perf_counter() - table_start:.2f}s")

v_nodes = pd.concat(node_frames, axis=0, ignore_index=True) if node_frames else pd.DataFrame(columns=['node_id', 'parent_node', 'node_name',  'node_type', 'workspace_id'])
v_nodes = v_nodes.drop_duplicates(subset=['node_id'], keep='first')
write_delta_table(v_nodes, 'v_nodes')
print(f"Created v_nodes with {len(v_nodes)} rows")

end_phase('build_nodes')

StatementMeta(, 2072f8f5-3a72-42ed-8bf2-6bfed48b828f, 13, Finished, Available, Finished, False)

Created v_nodes with 508 rows


In [10]:
notebookutils.session.stop()

StatementMeta(, 2072f8f5-3a72-42ed-8bf2-6bfed48b828f, 14, Finished, Available, Finished, False)